# 👥 تقسيم العملاء غير الموجّه (Customer Segmentation — Clustering)
### مشروع B6 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
فريق التسويق عايز يقسّم العملاء لـ **مجموعات متشابهة (Segments)** عشان يعمل لكل مجموعة حملة مخصّصة
بدل رسالة واحدة للكل. مفيش "إجابة صح" مسبقة → ده **تعلّم غير موجّه (Unsupervised)**.

**نوع المشكلة:** تجميع (Clustering) — اكتشاف بنية مخفية في البيانات بدون labels.

## 📦 ما الذي يثبته المشروع
**K-Means** + اختيار k (Elbow + **Silhouette**) · **DBSCAN** · **PCA** للتصوير ·
تحليل الشخصيات (Cluster Profiling) وتحويلها لتوصيات تسويقية.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/Ahmedelgendyyy/data-science-portfolio"
PROJECT = "ml/b6_customer_segmentation"          # مسار المشروع داخل portfolio/
PKGS    = []          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| التعلّم غير الموجّه | ISLR (ch.12) / Géron (ch.9) | لا توجد labels — نكتشف البنية |
| **K-Means + Inertia** | Géron (ch.9) | أشهر خوارزمية تجميع |
| اختيار k (Elbow + **Silhouette**) | Géron (ch.9) | كام مجموعة؟ السؤال الأصعب |
| التحجيم (Scaling) قبل التجميع | Géron (ch.9) | المسافات تتأثر بوحدات القياس |
| **DBSCAN** | Géron (ch.9) | تجميع حسب الكثافة + كشف الشواذ |
| **PCA** للتصوير | ISLR (ch.12) | رسم البيانات متعددة الأبعاد في 2D |

> 🎯 **بيُستخدم في الواقع:** تقسيم العملاء، التسويق المستهدف، أنظمة التوصية، ضغط الصور، اكتشاف المجتمعات.


## 0️⃣ المكتبات

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

ready ✓


## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [2]:
df = pd.read_csv('data/customer_segmentation_data.csv')
print('Shape:', df.shape)
features = ['age','annual_income_k','spending_score_1_100']
sns.pairplot(df[features]); plt.suptitle('Feature relationships', y=1.02); plt.show()
df[features].describe().T[['mean','std','min','max']]

Shape: (1000, 5)


C:\Users\DELL\AppData\Local\Temp\ipykernel_32752\810250255.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  sns.pairplot(df[features]); plt.suptitle('Feature relationships', y=1.02); plt.show()


,mean,std,min,max
age,35.607,11.051275,18.0,70.0
annual_income_k,62.397,27.718328,15.0,130.0
spending_score_1_100,59.497,24.801635,2.0,100.0


## 2️⃣ التحجيم (Scaling)

In [3]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(df[features])
print('Scaled shape:', X.shape)

Scaled shape: (1000, 3)


## 3️⃣ اختيار عدد المجموعات (Elbow + Silhouette)
- **Elbow:** نرسم الـ Inertia لكل k ونلاقي "الكوع".
- **Silhouette:** أعلى قيمة = أوضح فصل بين المجموعات.

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
inertias, sils = [], []
Ks = range(2, 11)
for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_); sils.append(silhouette_score(X, km.labels_))
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(list(Ks), inertias, 'o-'); ax[0].set_title('Elbow (Inertia)'); ax[0].set_xlabel('k')
ax[1].plot(list(Ks), sils, 'o-', color='green'); ax[1].set_title('Silhouette score'); ax[1].set_xlabel('k')
plt.show()
best_k = list(Ks)[int(np.argmax(sils))]
print('Best k by silhouette =', best_k)

Best k by silhouette = 3


C:\Users\DELL\AppData\Local\Temp\ipykernel_32752\2848242829.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4️⃣ تدريب K-Means وتحليل الشخصيات (Cluster Profiling)

In [5]:
km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X)
df['cluster'] = km.labels_
profile = df.groupby('cluster')[features].mean().round(1)
profile['size'] = df['cluster'].value_counts().sort_index()
print(profile)

          age  annual_income_k  spending_score_1_100  size
cluster                                                   
0        46.3             71.4                  34.2   406
1        31.4             85.4                  80.6   289
2        25.4             28.6                  73.1   305


## 5️⃣ المقارنة بـ DBSCAN (تجميع بالكثافة)
DBSCAN مش محتاج تحدّد عدد المجموعات، وبيكتشف الشواذ (نقطة فئتها -1).

In [6]:
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=0.4, min_samples=8).fit(X)
n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f'DBSCAN found {n_clusters} clusters + {(db.labels_==-1).sum()} outliers')

DBSCAN found 3 clusters + 96 outliers


## 6️⃣ تصوير المجموعات بـ PCA (2D)

In [7]:
from sklearn.decomposition import PCA
emb = PCA(n_components=2).fit_transform(X)
plt.figure(figsize=(8,6))
sns.scatterplot(x=emb[:,0], y=emb[:,1], hue=df['cluster'], palette='tab10', s=30)
plt.title('Customer Segments (PCA projection)'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_32752\1599167399.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Customer Segments (PCA projection)'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()


## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **النتيجة:** قسّمنا العملاء لـ مجموعات واضحة (k المختار بالـ Silhouette)، وكل مجموعة لها بروفايل مميّز.
- **أمثلة شخصيات (حسب البروفايل):** "دخل عالي/إنفاق عالي" = عملاء VIP · "دخل عالي/إنفاق منخفض" = فرصة ضائعة · "دخل منخفض/إنفاق عالي" = حسّاسون للعروض.
- **DBSCAN:** أعطى رؤية مختلفة + رصد شواذ ممكن يكونوا حالات خاصة.
- **التوصية:** حملة تسويقية مخصّصة لكل مجموعة (عروض VIP، تحفيز الإنفاق، برامج ولاء).
- **الخطوة القادمة:** إضافة ميزات سلوكية (تكرار الشراء)، وربط المجموعات بمعدّل التحويل.

> ✅ **اللي اتعلمته:** Scaling، K-Means، Elbow/Silhouette، DBSCAN، PCA، وتحليل الشخصيات.
